# Nairobi Urban Flood Digital Twin — Model Training

Trains the flood-forecast U-Net on Colab's GPU.

**Task.** Predict flood extent over days *t..t+2* from rainfall and terrain, on storm seasons the model has never seen.

Two models are trained on identical labels and splits, differing only in their inputs:

| | inputs | channels | what it measures |
|---|---|---|---|
| **A - forecast** | rainfall *t-7..t-1*, seasonality, 14/30-day totals, terrain | 17 | how well a storm can be *anticipated* |
| **B - NWP-driven** | the above **+ rainfall over *t..t+2*** | 20 | how well rainfall maps to flood extent |

Model A withholds day *t* rainfall, so the label cannot be recovered by summing an input channel. Model B receives it, as an operational system receives a forecast from a weather service — it is a rainfall-to-extent mapping, **not** a weather forecast, and must be reported as such.

**Runtime.** A few minutes each on a T4. Each dataset is ~1.9 MB and lives on the GPU, so there is no DataLoader, no memory-mapping, and system RAM is not involved.

Before running: set `REPO_URL` in the next cell, and enable the GPU via **Runtime -> Change runtime type -> T4 GPU**.

Read `LIMITATIONS.md` in the repo before quoting any number from this notebook in the thesis.

## 1. Environment

In [ ]:
REPO_URL = "https://github.com/eoringe/nairobi-flood-digital-twi.git"
BRANCH   = "main"

import torch, subprocess
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device      :", torch.cuda.get_device_name(0))
    print("VRAM        : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("\n!! No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.")

In [ ]:
# Optional: mount Drive to persist outputs across a runtime disconnect.
# The dataset does NOT come from Drive -- it ships in the repo.
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("\nDrive mounted. Outputs will be copied there in section 5.")
else:
    print("Skipping Drive. Outputs stay in the Colab runtime and are lost on disconnect.")

## 2. Get the code and data

The dataset is committed to the repository (~1.8 MB), so cloning brings it. No upload step.

It is small because rainfall is stored as 7 scalars per sample and terrain as one shared array, rather than the earlier format's 77 channels that were 91% duplicated copies — that version was 6 GB and repeatedly exhausted RAM.

In [ ]:
import os, shutil
os.chdir('/content')
if os.path.isdir('nairobi-flood-digital-twi'):
    shutil.rmtree('nairobi-flood-digital-twi')

!git clone --branch $BRANCH --single-branch $REPO_URL 2>&1 | tail -5
os.chdir('/content/nairobi-flood-digital-twi')

!pip install -q torch numpy matplotlib
print("\nWorking directory:", os.getcwd())

## 3. Verify the dataset

In [ ]:
import numpy as np, os, json

DATASET = 'data/processed/arrays/segmentation_dataset_v2_forecast_drainage.npz'
if not os.path.exists(DATASET):
    raise FileNotFoundError(
        DATASET + " not found.\n"
        "Rebuild locally with:  python -m src.ingestion.build_segmentation_dataset_v2\n"
        "then commit, push, and re-run the clone cell."
    )

d = np.load(DATASET, allow_pickle=False)
print("Dataset OK  (%.1f MB)\n" % (os.path.getsize(DATASET) / 1e6))
print("  rain_seq :", d['rain_seq'].shape, "  antecedent rainfall, mm/day")
print("  static   :", d['static'].shape, " ", list(d['channel_names'][:6]))
print("  y        :", d['y'].shape, " dtype =", d['y'].dtype)
print()
print("  train / val / test : %d / %d / %d samples"
      % (len(d['train_idx']), len(d['val_idx']), len(d['test_idx'])))
print("  positive pixel rate: %.2f%%" % (100 * d['y'].mean()))
print("  distinct masks     : %d" % len({y.tobytes() for y in d['y']}))
print()
print("  label parameters:")
for k, v in json.loads(str(d['params'][0])).items():
    print("    %-20s %s" % (k, v))

## 4. Train

Four runs: two label definitions x two input sets. Identical splits and architecture throughout, so every difference comes from the labels or the inputs.

**Label definitions**

| | how flood location is decided | spatial validation |
|---|---|---|
| **terrain** | low, flat ground near streams (HAND) | fails - no better than chance |
| **drainage** | built-up land on flat ground beside a channel | passes at every radius |

The drainage labels are the ones to report. Terrain is kept so the comparison can be shown rather than only the working version.

**Input sets**

| | rainfall it sees | expected F1 |
|---|---|---|
| **A - forecast** | only days *t-7..t-1* - must anticipate the storm | 0.15 - 0.25 |
| **B - NWP-driven** | also days *t..t+2*, as from a weather service | 0.85 - 0.95 |

Roughly 5-10 minutes each on a T4. Watch that recall never sits at 0.000 - that was the old collapse signature.

In [ ]:
# Model A / drainage labels  <-- the headline forecasting result
!python -m src.models.train_segmentation_v2     --data data/processed/arrays/segmentation_dataset_v2_forecast_drainage.npz     --tag forecast_drainage --epochs 60 --batch-size 16 --base 32 --lr 1e-3

In [ ]:
# Model B / drainage labels  <-- the deployed system
!python -m src.models.train_segmentation_v2     --data data/processed/arrays/segmentation_dataset_v2_nwp_drainage.npz     --tag nwp_drainage --epochs 60 --batch-size 16 --base 32 --lr 1e-3

In [ ]:
# Model A / terrain labels  (comparison only)
!python -m src.models.train_segmentation_v2     --data data/processed/arrays/segmentation_dataset_v2_forecast.npz     --tag forecast_terrain --epochs 60 --batch-size 16 --base 32 --lr 1e-3

In [ ]:
# Model B / terrain labels  (comparison only)
!python -m src.models.train_segmentation_v2     --data data/processed/arrays/segmentation_dataset_v2_nwp.npz     --tag nwp_terrain --epochs 60 --batch-size 16 --base 32 --lr 1e-3

## 5. Results

In [ ]:
import json, os, shutil

RUNS = [('forecast_drainage', 'A forecast', 'drainage'),
        ('nwp_drainage',      'B NWP-driven', 'drainage'),
        ('forecast_terrain',  'A forecast', 'terrain'),
        ('nwp_terrain',       'B NWP-driven', 'terrain')]

print('=' * 72)
print('TEST RESULTS - held-out storm seasons, never seen in training')
print('=' * 72)
print('%-16s %-10s %8s %8s %10s %8s' % ('model','labels','F1','IoU','precision','recall'))
print('-' * 72)
print('%-16s %-10s %8.4f %8s %10s %8s' % ('baseline stencil','-',0.1434,'-','0.0798','0.7021'))
print('%-16s %-10s %8.4f %8s %10s %8s' % ('baseline logreg','-',0.1592,'-','0.1673','0.1519'))
found = {}
for tag, model, labels in RUNS:
    f = f'models/time_series/segmentation_metrics_v2_{tag}.json'
    if not os.path.exists(f):
        print('%-16s %-10s %8s   (not trained yet)' % (model, labels, '-')); continue
    t = json.load(open(f))['test_metrics']; found[tag] = t
    print('%-16s %-10s %8.4f %8.4f %10.4f %8.4f'
          % (model, labels, t['f1'], t['iou'], t['precision'], t['recall']))
print('=' * 72)

if 'nwp_drainage' in found and 'forecast_drainage' in found:
    gap = found['nwp_drainage']['f1'] - found['forecast_drainage']['f1']
    print()
    print('Knowing the rainfall is worth %+.3f F1 (drainage labels).' % gap)
    print('That gap is the meteorological bottleneck: flood mapping is the easy part.')

if USE_DRIVE:
    out = '/content/drive/MyDrive/nairobi-flood-data/training-outputs'
    os.makedirs(out, exist_ok=True)
    for f in os.listdir('models/time_series'):
        if f.startswith('segmentation_'):
            shutil.copy(f'models/time_series/{f}', f'{out}/{f}')
    print()
    print('Saved to Drive:', out)

## 6. Example predictions

Generates side-by-side figures for the thesis Results chapter: terrain susceptibility, the label, and the model's predicted probability for held-out test storms.

Inspect these rather than trusting F1 alone. Predicted flooding should follow drainage lines and low-lying ground. If it looks like uniform blanket coverage, the model has learned the storm trigger but not the spatial distribution.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, torch, json

from src.models.train_segmentation_v2 import UNet, GpuDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
d = np.load('data/processed/arrays/segmentation_dataset_v2_forecast_drainage.npz', allow_pickle=False)

test = GpuDataset(d, d['test_idx'], device)

model = UNet(in_ch=test.in_channels, base=32).to(device)
model.load_state_dict(torch.load('models/time_series/segmentation_model_v2_forecast_drainage.pth',
                                 map_location=device))
model.eval()

susc = d['susceptibility']
dates = d['dates'][d['test_idx']]
events = d['event_ids'][d['test_idx']]

# Pick the largest-extent storm from each distinct test event. Taking the global
# top-N instead would return consecutive days of the same storm, which look
# nearly identical and waste a figure panel.
extents = np.array([d['y'][j].mean() for j in d['test_idx']])
picks = []
for ev in sorted(set(events)):
    cand = np.where((events == ev) & (extents > 0))[0]
    if len(cand):
        picks.append(int(cand[np.argmax(extents[cand])]))
picks = sorted(picks, key=lambda i: -extents[i])[:6]
print("Showing %d storms from %d held-out seasons\n" % (len(picks), len(set(events[picks]))))

fig, axes = plt.subplots(len(picks), 3, figsize=(11, 3.1 * len(picks)))
for row, i in enumerate(picks):
    with torch.no_grad():
        x, y = test.batch(torch.tensor([i], device=device))
        prob = torch.sigmoid(model(x))[0, 0].cpu().numpy()
    truth = y[0, 0].cpu().numpy()

    for ax, img, title, cmap in [
        (axes[row, 0], susc, 'Terrain susceptibility', 'terrain_r'),
        (axes[row, 1], truth, 'Label (extent %.1f%%)' % (100 * truth.mean()), 'Blues'),
        (axes[row, 2], prob, 'Predicted probability', 'Blues'),
    ]:
        im = ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
        ax.set_title(title, fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    axes[row, 0].set_ylabel('%s\n%s' % (dates[i], events[i]), fontsize=8)

plt.tight_layout()
plt.savefig('models/time_series/example_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: models/time_series/example_predictions.png")

if USE_DRIVE:
    import shutil
    shutil.copy('models/time_series/example_predictions.png',
                '/content/drive/MyDrive/nairobi-flood-data/training-outputs/')
    print("Copied to Drive.")